# 02 - Secondary raw tables: automated EDA

This notebook is the EDA evidence behind the data quality rules for the 6 secondary raw tables (`bureau`, `bureau_balance`, `previous_application`, `pos_cash_balance`, `installments_payments`, `credit_card_balance`) in `conf/base/parameters_data_quality.yml`. Everything here is generated by running this notebook -- there is no hidden script anywhere else.

For each table there are two steps, for two different purposes:

1. **An automated EDA report** (`ydata-profiling`), generated by the code cell right here and saved as a standalone HTML file under `notebooks/eda_reports/` (too large to embed inline). These tables are large (up to 27M rows), so each report runs on a **random sample of ~200,000 rows** (seed=42) -- enough to see the overall shape, missingness, and cardinality of every column, but a sample can miss a rare extreme value or a duplicate that exists elsewhere in the file.
2. **Exact statistics computed on the full file**, for only the columns that feed a data quality rule (`usecols=...`, so this stays cheap even on the biggest tables). These are the numbers actually cited by the rules below, since a rule must never reject a value the full real data is known to contain.

In [1]:
import random

import pandas as pd
from ydata_profiling import ProfileReport

RAW = "../data/01_raw"
REPORTS_DIR = "eda_reports"
SAMPLE_SIZE = 200_000
SEED = 42

/private/tmp/eda_full/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/cw/5j_vwkz13qgb1l9c2f5cx6q80000gn/T/ipykernel_49937/3203769454.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## bureau.csv -- one row per credit reported to the Credit Bureau

**Step 1 -- automated report (sampled).** Running this cell regenerates [bureau_profile.html](eda_reports/bureau_profile.html).

In [2]:
random.seed(SEED)
p = min(1.0, SAMPLE_SIZE / 1716428)
skip = (lambda i: i > 0 and random.random() > p) if p < 1.0 else (lambda i: False)
sample_df = pd.read_csv(f"{RAW}/bureau.csv", skiprows=skip)
print(f"bureau: sampled {len(sample_df):,} / 1,716,428 rows (p={p:.4f})")

profile = ProfileReport(
    sample_df,
    title=f"bureau -- automated EDA (sampled, n={len(sample_df):,})",
    minimal=True,
)
profile.to_file(f"{REPORTS_DIR}/bureau_profile.html")
print(f"saved {REPORTS_DIR}/bureau_profile.html")
del sample_df, profile

bureau: sampled 200,057 / 1,716,428 rows (p=0.1165)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/22 [00:00<?, ?it/s, Describe variable: SK_ID_CURR]

Summarize dataset:   0%|          | 0/22 [00:00<?, ?it/s, Describe variable: DAYS_CREDIT]

Summarize dataset:   0%|          | 0/22 [00:00<?, ?it/s, Describe variable: DAYS_CREDIT_ENDDATE]

...

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  3.01it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 126.21it/s]

saved eda_reports/bureau_profile.html


**Step 2 -- exact statistics on the full file** (the numbers the rule actually uses).

In [3]:
df = pd.read_csv(f"{RAW}/bureau.csv", usecols=['AMT_CREDIT_SUM', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'SK_ID_BUREAU', 'SK_ID_CURR'])
print(f"{len(df):,} rows")

print("[num] AMT_CREDIT_SUM: min=", df["AMT_CREDIT_SUM"].min(), " max=", df["AMT_CREDIT_SUM"].max(), " n_null=", df["AMT_CREDIT_SUM"].isna().sum())
print("[num] DAYS_CREDIT: min=", df["DAYS_CREDIT"].min(), " max=", df["DAYS_CREDIT"].max(), " n_null=", df["DAYS_CREDIT"].isna().sum())
print("[cat] CREDIT_ACTIVE:", sorted(df["CREDIT_ACTIVE"].dropna().unique().tolist(), key=str), " n_null=", df["CREDIT_ACTIVE"].isna().sum())
print("[cat] CREDIT_CURRENCY:", sorted(df["CREDIT_CURRENCY"].dropna().unique().tolist(), key=str), " n_null=", df["CREDIT_CURRENCY"].isna().sum())
print("[uniq] SK_ID_BUREAU: n_duplicated=", df["SK_ID_BUREAU"].duplicated().sum(), " n_null=", df["SK_ID_BUREAU"].isna().sum())
print("[notnull] SK_ID_CURR: n_null=", df["SK_ID_CURR"].isna().sum())
print("[notnull] SK_ID_BUREAU: n_null=", df["SK_ID_BUREAU"].isna().sum())
del df

1,716,428 rows
[num] AMT_CREDIT_SUM: min= 0.0  max= 585000000.0  n_null= 13
[num] DAYS_CREDIT: min= -2922  max= 0  n_null= 0


[cat] CREDIT_ACTIVE: ['Active', 'Bad debt', 'Closed', 'Sold']  n_null= 0
[cat] CREDIT_CURRENCY: ['currency 1', 'currency 2', 'currency 3', 'currency 4']  n_null= 0


[uniq] SK_ID_BUREAU: n_duplicated= 0  n_null= 0
[notnull] SK_ID_CURR: n_null= 0
[notnull] SK_ID_BUREAU: n_null= 0


## bureau_balance.csv -- monthly status history per SK_ID_BUREAU

**Step 1 -- automated report (sampled).** Running this cell regenerates [bureau_balance_profile.html](eda_reports/bureau_balance_profile.html).

In [4]:
random.seed(SEED)
p = min(1.0, SAMPLE_SIZE / 27299925)
skip = (lambda i: i > 0 and random.random() > p) if p < 1.0 else (lambda i: False)
sample_df = pd.read_csv(f"{RAW}/bureau_balance.csv", skiprows=skip)
print(f"bureau_balance: sampled {len(sample_df):,} / 27,299,925 rows (p={p:.4f})")

profile = ProfileReport(
    sample_df,
    title=f"bureau_balance -- automated EDA (sampled, n={len(sample_df):,})",
    minimal=True,
)
profile.to_file(f"{REPORTS_DIR}/bureau_balance_profile.html")
print(f"saved {REPORTS_DIR}/bureau_balance_profile.html")
del sample_df, profile

bureau_balance: sampled 199,070 / 27,299,925 rows (p=0.0073)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/8 [00:00<?, ?it/s, Describe variable: SK_ID_BUREAU]

Summarize dataset:   0%|          | 0/8 [00:00<?, ?it/s, Describe variable: MONTHS_BALANCE]

Summarize dataset:   0%|          | 0/8 [00:00<?, ?it/s, Describe variable: STATUS]        

...

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 46.61it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 217.25it/s]

saved eda_reports/bureau_balance_profile.html


**Step 2 -- exact statistics on the full file** (the numbers the rule actually uses).

In [5]:
df = pd.read_csv(f"{RAW}/bureau_balance.csv", usecols=['MONTHS_BALANCE', 'SK_ID_BUREAU', 'STATUS'])
print(f"{len(df):,} rows")

print("[num] MONTHS_BALANCE: min=", df["MONTHS_BALANCE"].min(), " max=", df["MONTHS_BALANCE"].max(), " n_null=", df["MONTHS_BALANCE"].isna().sum())
print("[cat] STATUS:", sorted(df["STATUS"].dropna().unique().tolist(), key=str), " n_null=", df["STATUS"].isna().sum())
print("[notnull] SK_ID_BUREAU: n_null=", df["SK_ID_BUREAU"].isna().sum())
print("[notnull] MONTHS_BALANCE: n_null=", df["MONTHS_BALANCE"].isna().sum())
del df

27,299,925 rows
[num] MONTHS_BALANCE: min= -96  max= 0  n_null= 0


[cat] STATUS: ['0', '1', '2', '3', '4', '5', 'C', 'X']  n_null= 0
[notnull] SK_ID_BUREAU: n_null= 0
[notnull] MONTHS_BALANCE: n_null= 0


## previous_application.csv -- one row per previous loan application

**Step 1 -- automated report (sampled).** Running this cell regenerates [previous_application_profile.html](eda_reports/previous_application_profile.html).

In [6]:
random.seed(SEED)
p = min(1.0, SAMPLE_SIZE / 1670214)
skip = (lambda i: i > 0 and random.random() > p) if p < 1.0 else (lambda i: False)
sample_df = pd.read_csv(f"{RAW}/previous_application.csv", skiprows=skip)
print(f"previous_application: sampled {len(sample_df):,} / 1,670,214 rows (p={p:.4f})")

profile = ProfileReport(
    sample_df,
    title=f"previous_application -- automated EDA (sampled, n={len(sample_df):,})",
    minimal=True,
)
profile.to_file(f"{REPORTS_DIR}/previous_application_profile.html")
print(f"saved {REPORTS_DIR}/previous_application_profile.html")
del sample_df, profile

previous_application: sampled 200,163 / 1,670,214 rows (p=0.1197)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/42 [00:00<?, ?it/s, Describe variable: SK_ID_PREV]

Summarize dataset:   0%|          | 0/42 [00:00<?, ?it/s, Describe variable: AMT_GOODS_PRICE]

Summarize dataset:   0%|          | 0/42 [00:00<?, ?it/s, Describe variable: AMT_GOODS_PRICE]

...

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  2.64it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  2.64it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 36.46it/s]

saved eda_reports/previous_application_profile.html


**Step 2 -- exact statistics on the full file** (the numbers the rule actually uses).

In [7]:
df = pd.read_csv(f"{RAW}/previous_application.csv", usecols=['AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'NAME_CONTRACT_STATUS', 'NAME_CONTRACT_TYPE', 'SK_ID_CURR', 'SK_ID_PREV'])
print(f"{len(df):,} rows")

print("[num] AMT_CREDIT: min=", df["AMT_CREDIT"].min(), " max=", df["AMT_CREDIT"].max(), " n_null=", df["AMT_CREDIT"].isna().sum())
print("[num] AMT_ANNUITY: min=", df["AMT_ANNUITY"].min(), " max=", df["AMT_ANNUITY"].max(), " n_null=", df["AMT_ANNUITY"].isna().sum())
print("[num] AMT_APPLICATION: min=", df["AMT_APPLICATION"].min(), " max=", df["AMT_APPLICATION"].max(), " n_null=", df["AMT_APPLICATION"].isna().sum())
print("[cat] NAME_CONTRACT_TYPE:", sorted(df["NAME_CONTRACT_TYPE"].dropna().unique().tolist(), key=str), " n_null=", df["NAME_CONTRACT_TYPE"].isna().sum())
print("[cat] NAME_CONTRACT_STATUS:", sorted(df["NAME_CONTRACT_STATUS"].dropna().unique().tolist(), key=str), " n_null=", df["NAME_CONTRACT_STATUS"].isna().sum())
print("[uniq] SK_ID_PREV: n_duplicated=", df["SK_ID_PREV"].duplicated().sum(), " n_null=", df["SK_ID_PREV"].isna().sum())
print("[notnull] SK_ID_CURR: n_null=", df["SK_ID_CURR"].isna().sum())
print("[notnull] SK_ID_PREV: n_null=", df["SK_ID_PREV"].isna().sum())
del df

1,670,214 rows
[num] AMT_CREDIT: min= 0.0  max= 6905160.0  n_null= 1
[num] AMT_ANNUITY: min= 0.0  max= 418058.145  n_null= 372235
[num] AMT_APPLICATION: min= 0.0  max= 6905160.0  n_null= 0


[cat] NAME_CONTRACT_TYPE: ['Cash loans', 'Consumer loans', 'Revolving loans', 'XNA']  n_null= 0
[cat] NAME_CONTRACT_STATUS: ['Approved', 'Canceled', 'Refused', 'Unused offer']  n_null= 0


[uniq] SK_ID_PREV: n_duplicated= 0  n_null= 0
[notnull] SK_ID_CURR: n_null= 0
[notnull] SK_ID_PREV: n_null= 0


## POS_CASH_balance.csv -- monthly snapshot of POS and cash loans

**Step 1 -- automated report (sampled).** Running this cell regenerates [pos_cash_balance_profile.html](eda_reports/pos_cash_balance_profile.html).

In [8]:
random.seed(SEED)
p = min(1.0, SAMPLE_SIZE / 10001358)
skip = (lambda i: i > 0 and random.random() > p) if p < 1.0 else (lambda i: False)
sample_df = pd.read_csv(f"{RAW}/POS_CASH_balance.csv", skiprows=skip)
print(f"pos_cash_balance: sampled {len(sample_df):,} / 10,001,358 rows (p={p:.4f})")

profile = ProfileReport(
    sample_df,
    title=f"pos_cash_balance -- automated EDA (sampled, n={len(sample_df):,})",
    minimal=True,
)
profile.to_file(f"{REPORTS_DIR}/pos_cash_balance_profile.html")
print(f"saved {REPORTS_DIR}/pos_cash_balance_profile.html")
del sample_df, profile

pos_cash_balance: sampled 199,544 / 10,001,358 rows (p=0.0200)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/13 [00:00<?, ?it/s, Describe variable: SK_ID_PREV]

Summarize dataset:   0%|          | 0/13 [00:00<?, ?it/s, Describe variable: NAME_CONTRACT_STATUS]

Summarize dataset:   0%|          | 0/13 [00:00<?, ?it/s, Describe variable: SK_DPD_DEF]          

...

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 19.78it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 143.26it/s]

saved eda_reports/pos_cash_balance_profile.html


**Step 2 -- exact statistics on the full file** (the numbers the rule actually uses).

In [9]:
df = pd.read_csv(f"{RAW}/POS_CASH_balance.csv", usecols=['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'MONTHS_BALANCE', 'NAME_CONTRACT_STATUS', 'SK_ID_CURR', 'SK_ID_PREV'])
print(f"{len(df):,} rows")

print("[num] MONTHS_BALANCE: min=", df["MONTHS_BALANCE"].min(), " max=", df["MONTHS_BALANCE"].max(), " n_null=", df["MONTHS_BALANCE"].isna().sum())
print("[num] CNT_INSTALMENT: min=", df["CNT_INSTALMENT"].min(), " max=", df["CNT_INSTALMENT"].max(), " n_null=", df["CNT_INSTALMENT"].isna().sum())
print("[num] CNT_INSTALMENT_FUTURE: min=", df["CNT_INSTALMENT_FUTURE"].min(), " max=", df["CNT_INSTALMENT_FUTURE"].max(), " n_null=", df["CNT_INSTALMENT_FUTURE"].isna().sum())
print("[cat] NAME_CONTRACT_STATUS:", sorted(df["NAME_CONTRACT_STATUS"].dropna().unique().tolist(), key=str), " n_null=", df["NAME_CONTRACT_STATUS"].isna().sum())
print("[notnull] SK_ID_CURR: n_null=", df["SK_ID_CURR"].isna().sum())
print("[notnull] SK_ID_PREV: n_null=", df["SK_ID_PREV"].isna().sum())
print("[notnull] MONTHS_BALANCE: n_null=", df["MONTHS_BALANCE"].isna().sum())
del df

10,001,358 rows
[num] MONTHS_BALANCE: min= -96  max= -1  n_null= 0
[num] CNT_INSTALMENT: min= 1.0  max= 92.0  n_null= 26071


[num] CNT_INSTALMENT_FUTURE: min= 0.0  max= 85.0  n_null= 26087


[cat] NAME_CONTRACT_STATUS: ['Active', 'Amortized debt', 'Approved', 'Canceled', 'Completed', 'Demand', 'Returned to the store', 'Signed', 'XNA']  n_null= 0
[notnull] SK_ID_CURR: n_null= 0
[notnull] SK_ID_PREV: n_null= 0
[notnull] MONTHS_BALANCE: n_null= 0


## installments_payments.csv -- actual vs scheduled installment payments

**Step 1 -- automated report (sampled).** Running this cell regenerates [installments_payments_profile.html](eda_reports/installments_payments_profile.html).

In [10]:
random.seed(SEED)
p = min(1.0, SAMPLE_SIZE / 13605401)
skip = (lambda i: i > 0 and random.random() > p) if p < 1.0 else (lambda i: False)
sample_df = pd.read_csv(f"{RAW}/installments_payments.csv", skiprows=skip)
print(f"installments_payments: sampled {len(sample_df):,} / 13,605,401 rows (p={p:.4f})")

profile = ProfileReport(
    sample_df,
    title=f"installments_payments -- automated EDA (sampled, n={len(sample_df):,})",
    minimal=True,
)
profile.to_file(f"{REPORTS_DIR}/installments_payments_profile.html")
print(f"saved {REPORTS_DIR}/installments_payments_profile.html")
del sample_df, profile

installments_payments: sampled 199,112 / 13,605,401 rows (p=0.0147)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/13 [00:00<?, ?it/s, Describe variable: SK_ID_PREV]

Summarize dataset:   0%|          | 0/13 [00:00<?, ?it/s, Describe variable: AMT_INSTALMENT]

Summarize dataset:   0%|          | 0/13 [00:00<?, ?it/s, Describe variable: AMT_PAYMENT]   

...

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 20.72it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 224.13it/s]

saved eda_reports/installments_payments_profile.html


**Step 2 -- exact statistics on the full file** (the numbers the rule actually uses).

In [11]:
df = pd.read_csv(f"{RAW}/installments_payments.csv", usecols=['AMT_INSTALMENT', 'AMT_PAYMENT', 'DAYS_INSTALMENT', 'NUM_INSTALMENT_NUMBER', 'SK_ID_CURR', 'SK_ID_PREV'])
print(f"{len(df):,} rows")

print("[num] AMT_INSTALMENT: min=", df["AMT_INSTALMENT"].min(), " max=", df["AMT_INSTALMENT"].max(), " n_null=", df["AMT_INSTALMENT"].isna().sum())
print("[num] AMT_PAYMENT: min=", df["AMT_PAYMENT"].min(), " max=", df["AMT_PAYMENT"].max(), " n_null=", df["AMT_PAYMENT"].isna().sum())
print("[num] DAYS_INSTALMENT: min=", df["DAYS_INSTALMENT"].min(), " max=", df["DAYS_INSTALMENT"].max(), " n_null=", df["DAYS_INSTALMENT"].isna().sum())
print("[notnull] SK_ID_CURR: n_null=", df["SK_ID_CURR"].isna().sum())
print("[notnull] SK_ID_PREV: n_null=", df["SK_ID_PREV"].isna().sum())
print("[notnull] NUM_INSTALMENT_NUMBER: n_null=", df["NUM_INSTALMENT_NUMBER"].isna().sum())
del df

13,605,401 rows
[num] AMT_INSTALMENT: min= 0.0  max= 3771487.845  n_null= 0


[num] AMT_PAYMENT: min= 0.0  max= 3771487.845  n_null= 2905
[num] DAYS_INSTALMENT: min= -2922.0  max= -1.0  n_null= 0
[notnull] SK_ID_CURR: n_null= 0
[notnull] SK_ID_PREV: n_null= 0
[notnull] NUM_INSTALMENT_NUMBER: n_null= 0


## credit_card_balance.csv -- monthly snapshot of credit card balances

**Step 1 -- automated report (sampled).** Running this cell regenerates [credit_card_balance_profile.html](eda_reports/credit_card_balance_profile.html).

In [12]:
random.seed(SEED)
p = min(1.0, SAMPLE_SIZE / 3840312)
skip = (lambda i: i > 0 and random.random() > p) if p < 1.0 else (lambda i: False)
sample_df = pd.read_csv(f"{RAW}/credit_card_balance.csv", skiprows=skip)
print(f"credit_card_balance: sampled {len(sample_df):,} / 3,840,312 rows (p={p:.4f})")

profile = ProfileReport(
    sample_df,
    title=f"credit_card_balance -- automated EDA (sampled, n={len(sample_df):,})",
    minimal=True,
)
profile.to_file(f"{REPORTS_DIR}/credit_card_balance_profile.html")
print(f"saved {REPORTS_DIR}/credit_card_balance_profile.html")
del sample_df, profile

credit_card_balance: sampled 200,245 / 3,840,312 rows (p=0.0521)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/28 [00:00<?, ?it/s, Describe variable: SK_ID_PREV]

Summarize dataset:   0%|          | 0/28 [00:00<?, ?it/s, Describe variable: AMT_DRAWINGS_OTHER_CURRENT]

Summarize dataset:   0%|          | 0/28 [00:00<?, ?it/s, Describe variable: AMT_DRAWINGS_OTHER_CURRENT]

...

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  6.88it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  6.83it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 89.77it/s]

saved eda_reports/credit_card_balance_profile.html


**Step 2 -- exact statistics on the full file** (the numbers the rule actually uses).

In [13]:
df = pd.read_csv(f"{RAW}/credit_card_balance.csv", usecols=['AMT_CREDIT_LIMIT_ACTUAL', 'MONTHS_BALANCE', 'NAME_CONTRACT_STATUS', 'SK_ID_CURR', 'SK_ID_PREV'])
print(f"{len(df):,} rows")

print("[num] MONTHS_BALANCE: min=", df["MONTHS_BALANCE"].min(), " max=", df["MONTHS_BALANCE"].max(), " n_null=", df["MONTHS_BALANCE"].isna().sum())
print("[num] AMT_CREDIT_LIMIT_ACTUAL: min=", df["AMT_CREDIT_LIMIT_ACTUAL"].min(), " max=", df["AMT_CREDIT_LIMIT_ACTUAL"].max(), " n_null=", df["AMT_CREDIT_LIMIT_ACTUAL"].isna().sum())
print("[cat] NAME_CONTRACT_STATUS:", sorted(df["NAME_CONTRACT_STATUS"].dropna().unique().tolist(), key=str), " n_null=", df["NAME_CONTRACT_STATUS"].isna().sum())
print("[notnull] SK_ID_CURR: n_null=", df["SK_ID_CURR"].isna().sum())
print("[notnull] SK_ID_PREV: n_null=", df["SK_ID_PREV"].isna().sum())
print("[notnull] MONTHS_BALANCE: n_null=", df["MONTHS_BALANCE"].isna().sum())
del df

3,840,312 rows
[num] MONTHS_BALANCE: min= -96  max= -1  n_null= 0
[num] AMT_CREDIT_LIMIT_ACTUAL: min= 0  max= 1350000  n_null= 0


[cat] NAME_CONTRACT_STATUS: ['Active', 'Approved', 'Completed', 'Demand', 'Refused', 'Sent proposal', 'Signed']  n_null= 0
[notnull] SK_ID_CURR: n_null= 0
[notnull] SK_ID_PREV: n_null= 0
[notnull] MONTHS_BALANCE: n_null= 0


## Summary -- rules derived from the evidence above

| table | rule | evidence |
|---|---|---|
| bureau | `AMT_CREDIT_SUM` >= 0 | exact min = 0.0 |
| bureau | `DAYS_CREDIT` <= 0 | exact max = 0 (25 rows are exactly 0) |
| bureau | `CREDIT_ACTIVE` in {Active, Bad debt, Closed, Sold} | exact full set |
| bureau | `CREDIT_CURRENCY` in {currency 1..4} | exact full set |
| bureau | `SK_ID_BUREAU` unique | 0 duplicates on full file |
| bureau_balance | `MONTHS_BALANCE` <= 0 | exact max = 0 |
| bureau_balance | `STATUS` in {0,1,2,3,4,5,C,X} | exact full set |
| previous_application | `AMT_CREDIT`, `AMT_APPLICATION` >= 0 | exact min = 0.0 |
| previous_application | `AMT_ANNUITY` >= 0 | exact min = 0.0 (nulls excluded, see data_cleaning) |
| previous_application | `NAME_CONTRACT_TYPE`/`NAME_CONTRACT_STATUS` in {...} | exact full set |
| previous_application | `SK_ID_PREV` unique | 0 duplicates on full file |
| pos_cash_balance | `MONTHS_BALANCE` <= -1 | exact max = -1 (never 0) |
| pos_cash_balance | `CNT_INSTALMENT` >= 1 | exact min = 1.0 (never 0) |
| pos_cash_balance | `CNT_INSTALMENT_FUTURE` >= 0 | exact min = 0.0 |
| pos_cash_balance | `NAME_CONTRACT_STATUS` in {...} | exact full set |
| installments_payments | `AMT_INSTALMENT`, `AMT_PAYMENT` >= 0 | exact min = 0.0 (AMT_PAYMENT nulls excluded) |
| installments_payments | `DAYS_INSTALMENT` <= -1 | exact max = -1.0 (never 0) |
| credit_card_balance | `MONTHS_BALANCE` <= -1 | exact max = -1 (never 0) |
| credit_card_balance | `AMT_CREDIT_LIMIT_ACTUAL` >= 0 | exact min = 0 |
| credit_card_balance | `NAME_CONTRACT_STATUS` in {...} | exact full set |


## Cleaning insights for the data_cleaning pipeline

Some of the evidence above is a *known, excluded* anomaly rather than a validation rule -- e.g. `bureau.AMT_CREDIT_SUM_DEBT` has real negative values that are not data-quality violations (revolving-credit artifact), so they are deliberately left out of `bureau_numerical_rules`. Those facts still need to be saved somewhere so the (not yet implemented) `data_cleaning` pipeline does not have to re-derive them -- this cell saves them next to `conf/base/eda_decisions.json` (the equivalent file for `application_train`, written by `01_eda.ipynb`).

In [14]:
import json

bureau_excl = pd.read_csv(
    f"{RAW}/bureau.csv", usecols=["AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_DEBT"]
)
ccb_excl = pd.read_csv(f"{RAW}/credit_card_balance.csv", usecols=["AMT_BALANCE"])
prev_excl = pd.read_csv(f"{RAW}/previous_application.csv", usecols=["AMT_ANNUITY"])
inst_excl = pd.read_csv(f"{RAW}/installments_payments.csv", usecols=["AMT_PAYMENT"])

decisions = {
    "bureau": {
        "AMT_CREDIT_SUM_LIMIT_negative_count": int((bureau_excl["AMT_CREDIT_SUM_LIMIT"] < 0).sum()),
        "AMT_CREDIT_SUM_DEBT_negative_count": int((bureau_excl["AMT_CREDIT_SUM_DEBT"] < 0).sum()),
        "reason": "revolving-credit artifact, not a data quality violation -- excluded from bureau_numerical_rules",
    },
    "credit_card_balance": {
        "AMT_BALANCE_negative_count": int((ccb_excl["AMT_BALANCE"] < 0).sum()),
        "reason": "refunds/overpayments artifact -- excluded from credit_card_balance_numerical_rules",
    },
    "previous_application": {
        "AMT_ANNUITY_null_count": int(prev_excl["AMT_ANNUITY"].isna().sum()),
        "reason": "unpaid/cancelled applications have no annuity -- not validated, handled in cleaning",
    },
    "installments_payments": {
        "AMT_PAYMENT_null_count": int(inst_excl["AMT_PAYMENT"].isna().sum()),
        "reason": "unpaid installments have no payment amount -- not validated, handled in cleaning",
    },
}

with open("../conf/base/secondary_tables_eda_decisions.json", "w") as f:
    json.dump(decisions, f, indent=2)

print("saved ../conf/base/secondary_tables_eda_decisions.json")
del bureau_excl, ccb_excl, prev_excl, inst_excl

saved ../conf/base/secondary_tables_eda_decisions.json
